# Tool atrophy pilot: does tool-RL erode the skill the tool replaces?

One model family, three checkpoints from the same base, evaluated **with the tool
turned off**: the base model, a tool-RL checkpoint (trained with a Python
interpreter), and a text-only RL checkpoint (same base, same kind of data, no
tool). Same prompt, same decoding, pass@k plus an arithmetic error rate.

| Readout | Meaning |
|---|---|
| tool-RL below text-RL at high k, or more arithmetic slips | **atrophy** |
| tool-RL above both at every k | **internalization** |
| tool-RL and text-RL overlap at every k | tool added nothing beyond RL, stop |

The only published hint is CIR (2505.24480) Fig. 6: one 7B model, no text-RL
control. This notebook adds the control and a direct skill metric.

**What runs.** `src/gen.py` samples one checkpoint under each condition with vLLM
and writes one JSONL per (model, condition) to Google Drive, checkpointing every
`CHUNK` problems, so a disconnect costs at most one chunk. Colab does GPU work
only: grading (math-verify) and the arithmetic check run on your laptop with
`src/grade.py`, and `src/report.py` makes the tables and figures
(`GRADE_ON_COLAB = True` grades here instead). Every model gets the same grid, so
each condition is read as its own three-way comparison.

| Condition | Prompt | If the model writes a code block |
|---|---|---|
| `prompt_off` | shared no-code prompt ("do not write code") | intercepted, denial message injected, nothing runs |
| `env_denied` | the model's tool prompt (asks for code) | intercepted, denial message injected, nothing runs |
| `tool_on` | the model's tool prompt | executed with `python -I`, 4-call budget, then the budget message |
| `plain_cot` (optional) | shared plain CoT prompt | nothing intercepted; the model fabricates its own output block. CIR's condition. |

**Hardware.** Pick a GPU runtime. A100 40 GB runs everything. L4 24 GB runs the
1.5B preset comfortably and the 7B preset slowly. T4 only fits 1.5B (fp16).
Rough wall-clock on an A100 with vLLM, MATH-500 at 16 samples/problem:

| Preset | per arm | whole grid (9 arms) |
|---|---|---|
| math-1.5b | 10 to 15 min | about 2 h |
| math-7b | 30 to 45 min | 5 to 7 h |

Run the **smoke test** first (about 5 minutes). It exercises every code path on
20 problems and tells you the pipeline is wired before you spend compute units.

In [ ]:
import shutil, subprocess
NO_GPU_MSG = ("NO GPU in this runtime. Colab: Runtime > Change runtime type > GPU (T4 / L4 / A100). "
              "VS Code Colab extension: disconnect and reconnect choosing a GPU runtime. Then run again from this cell.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print(NO_GPU_MSG)

## 1. Install

`vllm` pins its own torch. The kernel never imports torch or vllm (generation runs
in a subprocess), so no runtime restart is needed.

In [ ]:
import subprocess, sys
PKGS = "vllm math-verify datasets hf_transfer pandas matplotlib"
r = subprocess.run(f"pip install -q uv && uv pip install --system -q {PKGS}", shell=True)
if r.returncode != 0:
    print("uv failed, falling back to pip")
    subprocess.run(f"pip install -q {PKGS}", shell=True, check=True)
print(subprocess.run([sys.executable, "-c", "import vllm, math_verify, datasets; print('vllm', vllm.__version__)"],
                     capture_output=True, text=True).stdout or "import check failed, see above")

## 2. Where results go

Results are written under `239_project/results/<run>/<dataset>/` on Drive so they
survive the session. An existing `239_project` folder anywhere up to three levels
under MyDrive is reused (up to five levels deep), so you can file it wherever you like in Drive; otherwise
`MyDrive/239_project` is created. Accept the Drive popup; if the path printed below
starts with `/content/239_project_runs` the results are on the VM disk and die with
it. Outside Colab this falls back to a local folder.

In [ ]:
import glob, os
PROJECT_DIR = None   # None: reuse an existing 239_project folder anywhere up to 5 levels under MyDrive
                     # (so moving it around in Drive is fine), else create MyDrive/239_project.
                     # Or name it explicitly, e.g. "UCLA (Jun 2026 onward)/239_project".
try:
    from google.colab import drive  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if ON_COLAB:
    # If the auth popup is dismissed this raises: results must land on Drive, not on the VM disk.
    drive.mount("/content/drive")
    assert os.path.isdir("/content/drive/MyDrive"), "Drive did not mount; re-run this cell and accept the popup"
    if PROJECT_DIR:
        ROOT = f"/content/drive/MyDrive/{PROJECT_DIR}"
    else:
        hits = sorted((p for d in range(6) for p in glob.glob("/content/drive/MyDrive/" + "*/" * d + "239_project/results")
                       if os.path.isdir(p)), key=lambda p: p.count("/"))
        ROOT = os.path.dirname(hits[0]) if hits else "/content/drive/MyDrive/239_project"
        if len(hits) > 1:
            print("several 239_project/results folders on Drive, using the first:", *hits, sep="\n  ")
else:
    ROOT = os.path.abspath("239_project_runs")
RESULTS = f"{ROOT}/results"
os.makedirs(RESULTS, exist_ok=True)
WORK = "/content/239_project" if os.path.isdir("/content") else os.path.abspath("239_project_work")
os.makedirs(f"{WORK}/src", exist_ok=True)
os.chdir(WORK)
print("results ->", RESULTS)
print("work    ->", WORK)

## 3. Code

Three files, embedded from the repo's `src/`. Edit them in the repo and rebuild
the notebook with `python tools/build_notebook.py`; editing here works too for a
one-off run. `gen.py` prints its full CLI with `python src/gen.py --help`.

In [ ]:
%%writefile src/arith_check.py
"""Extract arithmetic equalities from a math chain of thought and check them exactly with Fractions.

How it works. LaTeX operators are normalised to Python ones. A *run* is a maximal stretch of
digits, operators, parentheses, decimal points and '=' signs. Each run is split on '=' into
segments; every adjacent pair where both segments evaluate and at least one contains an
operator is one checked equation. So the chain  7^4 + 5*7^3 = 2401 + 1715 = 4116  checks the
two steps (7^4 + 5*7^3, 2401 + 1715) and (2401 + 1715, 4116), and  2x + 3 = 7  is never checked:
the x breaks the run, and a segment glued to a letter, brace or backslash is discarded.

Everything is linear in the text length (character-class regexes, an ast walk per segment), so
a pathological sample cannot stall grading. Exponents are capped so 10^(10^10) is skipped.

check(text) -> (n_checked, n_wrong, details)  with details = [(lhs, rhs, lhs_value, rhs_value, ok), ...]
"""
import ast
import re
from fractions import Fraction

# Normalise common LaTeX / unicode operators to Python before scanning.
_NORM = [
    (r'\\times|\\cdot|×|·|\*\*?', '*'),
    (r'\\div|÷', '/'),
    (r'\\left\(|\\right\)', lambda m: '(' if 'left' in m.group(0) else ')'),
    (r'\\d?frac\{([^{}]+)\}\{([^{}]+)\}', r'((\1)/(\2))'),
    (r'\^\{([^{}]+)\}', r'^(\1)'),
    (r'\\,|\\!|\;|\\ ', ''),
    (r'\\%|%', ''),
    (r'\$', ''),
    (r'−|–', '-'),
    (r'\\approx|≈', '='),
    (r'(?<=\d),(?=\d{3}\b)', ''),      # thousands separators: 1,234 -> 1234
    (r'\.(?!\d)', ' ; '),             # a period not followed by a digit ends a sentence, not a number
    (r',', ' ; '),                    # any remaining comma is a list separator
]
_RUN = re.compile(r'[\d\s+\-*/^().=]+')
_SEG_START = re.compile(r'^[-(]?\s*[\d(]')
_SEG_END = re.compile(r'[\d)]$')
_HAS_OP = re.compile(r'[\d)]\s*[-+*/^]')
_GLUE = set('\\{}[]_')                 # a run touching one of these (or a letter/digit) is contaminated
MAX_EXP = 1000
MAX_BITS = 8192


def normalise(text):
    for pat, rep in _NORM:
        text = re.sub(pat, rep, text)
    return text


def _ev(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return Fraction(str(node.value)) if isinstance(node.value, float) else Fraction(node.value)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        v = _ev(node.operand)
        return -v if isinstance(node.op, ast.USub) else v
    if isinstance(node, ast.BinOp):
        a, b = _ev(node.left), _ev(node.right)
        if isinstance(node.op, ast.Add):
            return a + b
        if isinstance(node.op, ast.Sub):
            return a - b
        if isinstance(node.op, ast.Mult):
            return a * b
        if isinstance(node.op, ast.Div):
            return a / b                      # ZeroDivisionError propagates: the pair is skipped
        if isinstance(node.op, ast.Pow):
            if b.denominator != 1 or abs(b) > MAX_EXP:
                raise ValueError('exponent')
            if max(a.numerator.bit_length(), a.denominator.bit_length()) * abs(int(b)) > MAX_BITS:
                raise ValueError('too big')
            return a ** int(b)
    raise ValueError('unsupported')


def _value(seg):
    seg = seg.strip()
    if not seg or len(seg) > 400 or not _SEG_START.match(seg) or not _SEG_END.search(seg):
        raise ValueError('not an expression')
    return _ev(ast.parse(seg.replace('^', '**'), mode='eval').body)


def _ok(v, r, lhs, rhs, rel_tol):
    if v == r:
        return True
    try:
        dec = max((len(t.split('.')[1]) for t in re.findall(r'\d+\.\d+', lhs + ' ' + rhs)), default=0)
        if dec and abs(float(v) - float(r)) <= 0.5 * 10 ** -dec + 1e-12:
            return True
        return r != 0 and abs(float(v - r) / float(r)) <= rel_tol
    except OverflowError:
        return False


def _glued_before(prefix):
    """Is the text right before a run a math token (variable, subscript, \\command, brace)? Prose words
    such as 'So' or 'and' are not; a single letter, anything with a backslash, or a brace/bracket is."""
    if not prefix:
        return False
    if prefix[-1].isalnum() or prefix[-1] in _GLUE:
        return True
    w = re.search(r'(\S+)\s*$', prefix)
    w = w.group(1) if w else ''
    return bool(w) and ((len(w) == 1 and w.isalpha()) or w[-1] in _GLUE or '\\' in w)


def _glued_after(suffix):
    if not suffix:
        return False
    if suffix[0].isalnum() or suffix[0] in _GLUE:
        return True
    w = re.match(r'\s*(\S+)', suffix)
    w = w.group(1) if w else ''
    return bool(w) and ((len(w) == 1 and w.isalpha()) or w[0] in _GLUE or w.startswith('\\'))


def check(text, rel_tol=1e-6):
    text = normalise(text)
    n = wrong = 0
    details = []
    for m in _RUN.finditer(text):
        run = m.group(0)
        if '=' not in run:
            continue
        start = m.start() + (len(run) - len(run.lstrip()))
        end = m.end() - (len(run) - len(run.rstrip()))
        segs = run.split('=')
        if _glued_before(text[:start]):
            segs[0] = ''
        if _glued_after(text[end:]):
            segs[-1] = ''
        vals = []
        for seg in segs:
            try:
                vals.append((seg.strip(), _value(seg)))
            except Exception:  # noqa: BLE001  (syntax, unsupported node, zero division, huge power)
                vals.append(None)
        for (l, r) in zip(vals, vals[1:]):
            if l is None or r is None or not (_HAS_OP.search(l[0]) or _HAS_OP.search(r[0])):
                continue
            ok = _ok(l[1], r[1], l[0], r[0], rel_tol)
            n += 1
            wrong += (not ok)
            details.append((l[0], r[0], str(l[1]), str(r[1]), ok))
    return n, wrong, details


if __name__ == '__main__':
    for s in [
        "So 12 * 13 = 156 and 156 + 7 = 163.",
        "Wrong: 17 * 23 = 381. Also 100 - 37 = 73 is wrong too.",
        "$7^4 + 5 \\cdot 7^3 + 9 \\cdot 7^2 - 7 - 14 = 2401 + 1715 + 441 - 7 - 14 = 4536$",
        "Equation x + 3 = 5 should not match; 2x = 4 should not match; x - 3 = 7 neither.",
        "Rounded: 22/7 = 3.14 is fine; 1/3 = 0.333. Thus 4 + 4 = 8 = 2 * 4.",
    ]:
        n, w, d = check(s)
        print(f"n={n} wrong={w} :: {s}")
        for x in d:
            print("    ", x)

In [ ]:
%%writefile src/gen.py
#!/usr/bin/env python
"""Sample one model under one or more tool conditions with vLLM and write JSONL.

Every model gets the same grid. Conditions:
  prompt_off   shared no-code prompt ("do not write code"). If the model writes a
               code block anyway, the harness intercepts it and injects the denial
               message, so nothing is ever executed and no output is fabricated.
               n_calls on these rows = non-compliance count.
  env_denied   the model's tool prompt (asks for code). Every code block is
               intercepted and answered with the denial message.
  tool_on      the model's tool prompt. Code is executed with `python -I` in a
               temp dir (NOT a sandbox: fine for Colab, not for a shared box).
               Calls past --max-calls get the budget message instead.
  plain_cot    optional. Shared plain CoT prompt, one pass, no interception at
               all: the condition prior work (CIR) used. Code the model writes is
               just text and it will fabricate its own ```output block.

Only `main()` imports vllm, so the helpers can be unit-tested on a laptop.

Example
  python gen.py --model GAIR/ToRL-1.5B --tag tool_rl \
      --conditions prompt_off,env_denied,tool_on --native-template torl \
      --dataset math500 --n-problems 100 --n 16 --max-tokens 3000 \
      --max-model-len 4096 --out results/math-1.5b
"""
import argparse
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import arith_check  # noqa: E402

# --------------------------------------------------------------------------- prompts
# Verified against the sources on 2026-10-13:
#   qwen_math_cot  Oat-Zero model card (sail/Qwen2.5-Math-*-Oat-Zero), also the
#                  Qwen2.5-Math CoT prompt. Shared tool-off prompt for every model.
#   qwen_math_tir  Qwen2.5-Math TIR prompt (same words ToRL uses).
#   torl           ToRL paper Fig. 3, verbatim, raw User/Assistant format.
#   cir            CIR paper (2505.24480) Fig. 5 text, verbatim. The paper does not
#                  say whether it sits in the system or user turn; system assumed.
TEMPLATES = {
    "qwen_math_cot": (
        "<|im_start|>system\nPlease reason step by step, and put your final answer "
        "within \\boxed{}.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "qwen_math_nocode": (
        "<|im_start|>system\nPlease reason step by step, and put your final answer "
        "within \\boxed{}. Do not write code: solve the problem in natural language and "
        "arithmetic only.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "qwen_math_tir": (
        "<|im_start|>system\nPlease integrate natural language reasoning with programs to "
        "solve the problem above, and put your final answer within \\boxed{}.<|im_end|>\n"
        "<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
    "torl": (
        "A conversation between User and Assistant. The user asks a question, and the "
        "Assistant solves it.\nUser: Please integrate natural language reasoning with "
        "programs to solve the problem above, and put your final answer within \\boxed{}.\n"
        "{q}\nAssistant:"
    ),
    "cir": (
        "<|im_start|>system\nPlease solve the following problem step by step. During your "
        "reasoning process, if needed, you can choose to write python code to enhance your "
        "reasoning. The code executor will run your code and provide the execution results "
        "back to you to support your reasoning process. Please put the final answer within "
        "\\boxed{}.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    ),
}

# Injected in place of execution results. AdaTIR wording for the over-budget case;
# an honest "no interpreter" wording for prompt_off / env_denied (override: --denial-text).
DENIAL_BUDGET = ("Tool call budget exceeded; the code was not executed. "
                 "Please continue reasoning and give the final answer.")
DENIAL_UNAVAILABLE = ("No interpreter is available; the code was not executed. "
                      "Continue reasoning without running code and give the final answer.")
DENIAL = DENIAL_UNAVAILABLE

STOP_FENCE = "```\n"       # end of any fenced block
STOP_OUTPUT = "```output"  # model opens the output block itself
TEXT_STOPS = ["<|im_end|>", "<|endoftext|>", "\nUser:"]

CODE_RE = re.compile(r"```(?:python|py)[ \t]*\n(.*?)```", re.S)
FENCED_RE = re.compile(r"```.*?```", re.S)
CONDITIONS = ("prompt_off", "env_denied", "tool_on", "plain_cot")
LOOP_CONDITIONS = ("prompt_off", "env_denied", "tool_on")


def build_prompt(template: str, question: str) -> str:
    return TEMPLATES[template].replace("{q}", question)


def last_code_block(text: str):
    """Last closed ```python block in `text`, or None."""
    blocks = CODE_RE.findall(text)
    return blocks[-1] if blocks else None


def last_boxed(text: str):
    """Content of the last \\boxed{...} with balanced braces, or None."""
    i = text.rfind("\\boxed")
    if i < 0:
        return None
    j = text.find("{", i)
    if j < 0:
        return None
    depth = 0
    for k in range(j, len(text)):
        if text[k] == "{":
            depth += 1
        elif text[k] == "}":
            depth -= 1
            if depth == 0:
                return text[j + 1:k]
    return None


def strip_fenced(text: str) -> str:
    return FENCED_RE.sub(" ", text)


# --------------------------------------------------------------------------- grading
def make_grader():
    try:
        from math_verify import parse, verify  # type: ignore

        def grade(response: str, gold: str) -> bool:
            try:
                g = parse(gold if "\\boxed" in gold else f"${gold}$")
                p = parse(response)
                return bool(verify(g, p))
            except Exception:
                return False
        return grade, "math_verify"
    except ImportError:
        def norm(s):
            return re.sub(r"\s+|\\left|\\right|\\!|\\,|\$", "", s or "").replace("dfrac", "frac")

        def grade(response: str, gold: str) -> bool:
            b = last_boxed(response)
            return b is not None and norm(b) == norm(last_boxed(gold) or gold)
        return grade, "string"


# --------------------------------------------------------------------------- data
def load_problems(name: str, n_problems: int, seed: int):
    from datasets import load_dataset  # type: ignore
    if name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [dict(id=r["unique_id"], problem=r["problem"], answer=r["answer"],
                      subject=r["subject"], level=str(r["level"])) for r in ds]
    elif name == "aime24":
        ds = load_dataset("math-ai/aime24", split="test")
        items = [dict(id=f"aime24-{r['id']}", problem=r["problem"],
                      answer=last_boxed(r["solution"]) or r["solution"],
                      subject="aime24", level="aime") for r in ds]
    elif name == "aime25":
        ds = load_dataset("math-ai/aime25", split="test")
        items = [dict(id=f"aime25-{r['id']}", problem=r["problem"], answer=str(r["answer"]),
                      subject="aime25", level="aime") for r in ds]
    else:
        raise ValueError(f"unknown dataset {name}")
    if n_problems and n_problems < len(items):
        idx = sorted(random.Random(seed).sample(range(len(items)), n_problems))
        items = [items[i] for i in idx]
    return items


# --------------------------------------------------------------------------- executor
def execute(code: str, timeout: float = 10.0, max_chars: int = 2000):
    """Run `code` with python -I in a temp dir. Returns (output_text, errored)."""
    with tempfile.TemporaryDirectory() as d:
        path = Path(d) / "snippet.py"
        path.write_text(code)
        try:
            r = subprocess.run([sys.executable, "-I", str(path)], cwd=d, capture_output=True,
                               stdin=subprocess.DEVNULL, text=True, timeout=timeout)
            out, err = r.stdout, r.returncode != 0
            if err:
                tail = r.stderr.strip().splitlines()
                out = (out + "\n" + tail[-1]) if tail else out
        except subprocess.TimeoutExpired:
            out, err = f"Timeout: code ran longer than {timeout:.0f}s", True
    out = out.strip()
    if len(out) > max_chars:
        out = out[:max_chars] + "...[truncated]"
    return out, err


# --------------------------------------------------------------------------- tool loop
class Sample:
    __slots__ = ("pidx", "sidx", "prompt", "gen", "gen_tokens", "n_calls", "n_denied",
                 "n_exec_err", "n_injected", "rounds", "done", "truncated", "finish")

    def __init__(self, pidx, sidx, prompt):
        self.pidx, self.sidx, self.prompt = pidx, sidx, prompt
        self.gen = ""
        self.gen_tokens = 0
        self.n_calls = self.n_denied = self.n_exec_err = self.n_injected = self.rounds = 0
        self.done = self.truncated = False
        self.finish = None

    @property
    def context(self):
        return self.prompt + self.gen


def advance(s: Sample, text: str, n_tokens: int, finish_reason, stop_reason, cond: str,
            max_calls: int, executor=execute, denial=None):
    """Apply one generation chunk to sample `s`. Returns the text to inject (or None)."""
    s.gen += text
    s.gen_tokens += n_tokens
    s.rounds += 1
    if finish_reason == "length":
        s.done, s.truncated, s.finish = True, True, "length"
        return None
    if stop_reason not in (STOP_FENCE, STOP_OUTPUT):
        s.done, s.finish = True, "eos"
        return None
    code = last_code_block(text)
    if code is None:
        return None  # a non-python fence or a fabricated ```output: keep generating
    s.n_calls += 1
    if cond == "tool_on" and s.n_calls <= max_calls:
        result, err = executor(code)
        s.n_exec_err += int(err)
    else:
        result = DENIAL_BUDGET if cond == "tool_on" else (denial or DENIAL_UNAVAILABLE)
        s.n_denied += 1
    inj = ("\n" + result + "\n```\n") if stop_reason == STOP_OUTPUT else ("```output\n" + result + "\n```\n")
    s.gen += inj
    s.n_injected += 1
    return inj


TOOL_STOPS = (STOP_FENCE, STOP_OUTPUT)


def splice_round(pending, cond, args, pool):
    """pending: list of (sample, text, n_tokens, finish_reason, stop_reason) for one round.
    Executes tool_on code calls in parallel, then applies `advance` to every sample."""
    futures = {}
    for s, text, ntok, fin, stop in pending:
        if (cond == "tool_on" and stop in TOOL_STOPS and fin != "length"
                and s.n_calls + 1 <= args.max_calls):
            code = last_code_block(text)
            if code is not None:
                futures[id(s)] = pool.submit(execute, code, args.exec_timeout, args.max_output_chars)
    t0 = time.time()
    if futures:
        print(f"  executing {len(futures)} code blocks ({args.exec_workers} workers, "
              f"{args.exec_timeout:.0f}s timeout each)...", flush=True)
    for s, text, ntok, fin, stop in pending:
        fut = futures.get(id(s))
        if fut is not None:
            ex = lambda code, f=fut: f.result()  # noqa: E731
        else:
            ex = lambda code: execute(code, args.exec_timeout, args.max_output_chars)  # noqa: E731
        advance(s, text, ntok, fin, stop, cond, args.max_calls, executor=ex,
                denial=getattr(args, "denial_text", None))
    if futures:
        n_err = sum(1 for s, *_ in pending if id(s) in futures and futures[id(s)].result()[1])
        print(f"  executed in {time.time() - t0:.0f}s, {n_err} errored or timed out", flush=True)


def run_tool_loop(generate, tokenize_len, samples, cond, args, pool):
    """Rounds 2..N. generate(prompts, stops, budgets, seeds) -> [(text, n_tokens, finish, stop_reason)]."""
    max_rounds = 2 * args.max_calls + 6
    rnd, t0 = 1, time.time()
    while True:
        rnd += 1
        keep, budgets = [], []
        for s in samples:
            if s.done:
                continue
            remaining = min(args.max_tokens - s.gen_tokens,
                            args.max_model_len - tokenize_len(s.context) - 1)
            if remaining < 16:
                s.done, s.truncated, s.finish = True, True, "length"
            else:
                keep.append(s)
                budgets.append(remaining)
        if not keep:
            break
        final = any(s.rounds >= max_rounds for s in keep)
        stops = TEXT_STOPS if final else TEXT_STOPS + list(TOOL_STOPS)
        print(f"  round {rnd}: {len(keep)} of {len(samples)} samples still generating"
              f"{' (final round, no more tool stops)' if final else ''}, {time.time() - t0:.0f}s into the loop",
              flush=True)
        seeds = [(args.seed * 1_000_003 + s.pidx * 4096 + s.sidx * 64 + s.rounds) % (2 ** 31) for s in keep]
        outs = generate([s.context for s in keep], stops, budgets, seeds)
        splice_round([(s,) + tuple(o) for s, o in zip(keep, outs)], cond, args, pool)


# --------------------------------------------------------------------------- records
def to_record(s: Sample, item, cond, template, model, tag, grade=None):
    """One JSONL row. With grade=None (gen.py --no-grade) `correct`, `arith_n` and `arith_wrong` are
    left null for src/grade.py to fill on another machine; everything else is generation-time data."""
    text = s.gen
    fake_outputs = max(0, text.count(STOP_OUTPUT) - s.n_injected)
    if grade is None:
        correct = a_n = a_wrong = None
    else:
        correct = bool(grade(text, item["answer"]))
        a_n, a_wrong, _ = arith_check.check(strip_fenced(text))
    return dict(
        model=model, tag=tag, condition=cond, template=template,
        problem_id=item["id"], subject=item.get("subject"), level=item.get("level"),
        sample_idx=s.sidx, correct=correct, gold=item["answer"],
        boxed=last_boxed(text), boxed_present=last_boxed(text) is not None,
        truncated=s.truncated, finish=s.finish, gen_tokens=s.gen_tokens, rounds=s.rounds,
        code_emitted="```python" in text or "```py" in text, fake_outputs=fake_outputs,
        n_calls=s.n_calls, n_denied=s.n_denied, n_exec_err=s.n_exec_err,
        arith_n=a_n, arith_wrong=a_wrong, text=text,
    )


# --------------------------------------------------------------------------- main
def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", required=True)
    p.add_argument("--revision", default=None)
    p.add_argument("--tag", required=True, help="role label: base | tool_rl | text_rl | ...")
    p.add_argument("--conditions", default="prompt_off")
    p.add_argument("--shared-template", default="qwen_math_nocode",
                   help="no-code prompt used for prompt_off, same for every model")
    p.add_argument("--native-template", default="qwen_math_tir",
                   help="the model's tool prompt, used for env_denied and tool_on")
    p.add_argument("--plain-template", default="qwen_math_cot", help="prompt for plain_cot")
    p.add_argument("--denial-text", default=None,
                   help="override the message injected for prompt_off / env_denied")
    p.add_argument("--dataset", default="math500", choices=["math500", "aime24", "aime25"])
    p.add_argument("--n-problems", type=int, default=0, help="0 = all")
    p.add_argument("--n", type=int, default=16, help="samples per problem")
    p.add_argument("--max-tokens", type=int, default=3000)
    p.add_argument("--max-model-len", type=int, default=4096)
    p.add_argument("--allow-long", action="store_true",
                   help="override max_position_embeddings so max-model-len can exceed the config")
    p.add_argument("--temperature", type=float, default=0.6)
    p.add_argument("--top-p", type=float, default=0.95)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max-calls", type=int, default=4)
    p.add_argument("--exec-timeout", type=float, default=10.0)
    p.add_argument("--exec-workers", type=int, default=8)
    p.add_argument("--max-output-chars", type=int, default=2000)
    p.add_argument("--gpu-mem", type=float, default=0.90)
    p.add_argument("--dtype", default="auto")
    p.add_argument("--out", required=True, help="results directory")
    p.add_argument("--overwrite", action="store_true", help="redo finished arms and discard partial files")
    p.add_argument("--no-grade", action="store_true",
                   help="write raw trajectories only (correct/arith fields null); grade later with src/grade.py")
    p.add_argument("--chunk", type=int, default=100,
                   help="problems per checkpoint; records are appended to <arm>.jsonl.partial after each "
                        "chunk and a restart resumes from there (0 = whole arm in one go)")
    return p.parse_args(argv)


# --------------------------------------------------------------------------- checkpointing
# An arm is "done" when its .meta.json exists (written last). While an arm runs, each chunk of
# problems is appended to <arm>.jsonl.partial, so a disconnect loses at most one chunk. The
# final .jsonl is written atomically (tmp + rename) and the partial file is removed.
def partial_path(path: Path) -> Path:
    return Path(str(path) + ".partial")


def meta_path(path: Path) -> Path:
    return path.with_suffix(".meta.json")


def read_partial(path: Path, n: int):
    """Records already on disk for this arm. Keeps only problems with all n sample_idx present
    (last write wins), so a torn chunk is simply redone. Returns (records, finished problem ids)."""
    pp = partial_path(path)
    if not pp.exists():
        return [], set()
    by_pid = {}
    with open(pp) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue  # torn last line from a crash mid-write
            by_pid.setdefault(r["problem_id"], {})[r["sample_idx"]] = r
    done = {pid for pid, d in by_pid.items() if len(d) == n}
    recs = [by_pid[pid][k] for pid in done for k in sorted(by_pid[pid])]
    return recs, done


def plan_chunks(items, done_ids, chunk):
    """Remaining (global_index, item) pairs, grouped into chunks. Global indices are kept so
    per-sample seeds do not depend on chunking or on what was already finished."""
    remaining = [(i, it) for i, it in enumerate(items) if it["id"] not in done_ids]
    size = chunk if chunk > 0 else max(1, len(remaining))
    return [remaining[s:s + size] for s in range(0, len(remaining), size)]


def write_jsonl(path: Path, recs, mode="a"):
    # A crash mid-write leaves a torn last line with no newline; start the append on a fresh line
    # so the torn line stays one (ignorable) bad record instead of swallowing the next good one.
    if mode == "a" and path.exists() and path.stat().st_size > 0:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            needs_nl = f.read(1) != b"\n"
        if needs_nl:
            with open(path, "a") as f:
                f.write("\n")
    with open(path, mode) as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass  # FUSE mounts such as Google Drive may not support fsync


def write_jsonl_atomic(path: Path, recs):
    tmp = Path(str(path) + ".tmp")
    write_jsonl(tmp, recs, mode="w")
    os.replace(tmp, path)


def main(argv=None):
    args = parse_args(argv)
    conds = [c.strip() for c in args.conditions.split(",") if c.strip()]
    for c in conds:
        if c not in CONDITIONS:
            sys.exit(f"unknown condition {c}; choose from {CONDITIONS}")
    out_dir = Path(args.out) / args.dataset
    out_dir.mkdir(parents=True, exist_ok=True)

    def out_path(cond, template):
        return out_dir / f"{args.tag}__{cond}__{template}.jsonl"

    def template_for(cond):
        if cond == "prompt_off":
            return args.shared_template
        if cond == "plain_cot":
            return args.plain_template
        return args.native_template

    todo = [(c, template_for(c)) for c in conds
            if args.overwrite or not meta_path(out_path(c, template_for(c))).exists()]
    if not todo:
        print("all conditions already done, nothing to do (use --overwrite to redo)")
        return
    print(f"gen.py starting: {args.tag} {args.model}; todo={todo}", flush=True)
    print(f"loading {args.dataset} from the HF hub...", flush=True)
    items = load_problems(args.dataset, args.n_problems, args.seed)
    if args.no_grade:
        grade, grader_name = None, "none (grade with src/grade.py)"
    else:
        print("loading grader (math_verify)...", flush=True)
        grade, grader_name = make_grader()
    print(f"{len(items)} problems from {args.dataset}; grader={grader_name}", flush=True)
    print("importing torch + vllm (first time: a minute or two)...", flush=True)

    os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
    # The FlashInfer sampler JIT-builds a kernel on first use (about two minutes on a fresh Colab
    # runtime, every time, since the cache dies with the VM). The torch sampler needs no build.
    os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
    if args.allow_long:
        os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "1"
    import torch  # type: ignore
    if not torch.cuda.is_available():
        sys.exit("no CUDA device is visible: this runtime has no GPU. Colab: Runtime > Change runtime type > GPU "
                 "(VS Code: reconnect choosing a GPU runtime), then run again from the first cell.")
    from vllm import LLM, SamplingParams  # type: ignore
    dtype = args.dtype
    if dtype == "auto":
        major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 8
        dtype = "bfloat16" if major >= 8 else "float16"
    kw = dict(model=args.model, dtype=dtype, max_model_len=args.max_model_len, seed=args.seed,
              gpu_memory_utilization=args.gpu_mem, enable_prefix_caching=True)
    if args.revision:
        kw["revision"] = args.revision
    if args.allow_long:
        kw["hf_overrides"] = {"max_position_embeddings": args.max_model_len}
    t0 = time.time()
    llm = LLM(**kw)
    tok = llm.get_tokenizer()
    print(f"loaded {args.model} ({dtype}) in {time.time() - t0:.0f}s", flush=True)

    def tokenize_len(text):
        return len(tok(text, add_special_tokens=False)["input_ids"])

    def sp(n, max_tokens, stops, seed=None):
        return SamplingParams(n=n, temperature=args.temperature, top_p=args.top_p,
                              seed=args.seed if seed is None else seed,
                              max_tokens=max_tokens, stop=stops, include_stop_str_in_output=True)

    def generate(prompts, stops, budgets, seeds):
        params = [sp(1, b, stops, seed) for b, seed in zip(budgets, seeds)]
        res = llm.generate(prompts, params, use_tqdm=True)
        outs = []
        for r in res:
            o = r.outputs[0]
            outs.append((o.text, len(o.token_ids), o.finish_reason, o.stop_reason))
        return outs

    def run_arm(part, cond, template):
        """Generate every sample for the (global_index, item) pairs in `part`; returns Samples."""
        prompts = [build_prompt(template, it["problem"]) for _, it in part]
        samples = []
        if cond == "plain_cot":
            res = llm.generate(prompts, sp(args.n, args.max_tokens, TEXT_STOPS), use_tqdm=True)
            for (pidx, _), prompt, r in zip(part, prompts, res):
                for sidx, o in enumerate(r.outputs):
                    s = Sample(pidx, sidx, prompt)
                    advance(s, o.text, len(o.token_ids), o.finish_reason, None, cond, 0)
                    if o.finish_reason == "length":
                        s.truncated, s.finish = True, "length"
                    samples.append(s)
            return samples
        res = llm.generate(prompts, sp(args.n, args.max_tokens, TEXT_STOPS + list(TOOL_STOPS)), use_tqdm=True)
        first = []
        for (pidx, _), prompt, r in zip(part, prompts, res):
            for sidx, o in enumerate(r.outputs):
                s = Sample(pidx, sidx, prompt)
                samples.append(s)
                first.append((s, o.text, len(o.token_ids), o.finish_reason, o.stop_reason))
        pool = ThreadPoolExecutor(max_workers=args.exec_workers)
        splice_round(first, cond, args, pool)
        run_tool_loop(generate, tokenize_len, samples, cond, args, pool)
        pool.shutdown(wait=True)
        return samples

    order = {it["id"]: i for i, it in enumerate(items)}
    for cond, template in todo:
        t1 = time.time()
        path = out_path(cond, template)
        if args.overwrite:
            partial_path(path).unlink(missing_ok=True)
        kept, _ = read_partial(path, args.n)
        kept = [r for r in kept if r["problem_id"] in order]
        done_ids = {r["problem_id"] for r in kept}
        chunks = plan_chunks(items, done_ids, args.chunk)
        print(f"== {args.tag} / {cond} / {template}: {len(items)} problems x {args.n} samples"
              + (f"; resuming, {len(done_ids)} problems already on disk" if done_ids else "")
              + f"; {len(chunks)} chunk(s) to run", flush=True)
        new_recs = []
        for ci, part in enumerate(chunks, 1):
            tc = time.time()
            samples = run_arm(part, cond, template)
            tg = time.time()
            if grade is not None:
                print(f"   grading {len(samples)} samples (math_verify, serial)...", flush=True)
            recs = []
            for i, smp in enumerate(samples, 1):
                recs.append(to_record(smp, items[smp.pidx], cond, template, args.model, args.tag, grade))
                if grade is not None and i % 40 == 0:
                    print(f"   graded {i}/{len(samples)}, {time.time() - tg:.0f}s", flush=True)
            print(f"   writing {len(recs)} records to {partial_path(path).name}...", flush=True)
            tw = time.time()
            write_jsonl(partial_path(path), recs)
            new_recs += recs
            acc_txt = (f"pass@1={sum(r['correct'] for r in recs) / max(1, len(recs)):.3f}" if grade is not None
                       else "ungraded")
            print(f"   chunk {ci}/{len(chunks)}: {len(part)} problems, {acc_txt}, "
                  f"{(time.time() - tc) / 60:.1f} min (write {time.time() - tw:.0f}s), checkpointed", flush=True)
        gen_s = time.time() - t1
        recs = kept + new_recs
        recs.sort(key=lambda r: (order[r["problem_id"]], r["sample_idx"]))
        print(f"   writing {path.name} + meta...", flush=True)
        write_jsonl_atomic(path, recs)
        graded = [r["correct"] for r in recs if r.get("correct") is not None]
        acc = sum(graded) / len(graded) if graded and len(graded) == len(recs) else None
        meta = dict(vars(args), condition=cond, template=template, dtype=dtype, grader=grader_name,
                    graded=acc is not None, n_problems=len(items), n_records=len(recs), n_resumed=len(done_ids),
                    pass1=acc, gen_seconds=gen_s, total_gen_tokens=sum(r["gen_tokens"] for r in recs),
                    finished_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))
        meta_path(path).write_text(json.dumps(meta, indent=1))
        partial_path(path).unlink(missing_ok=True)
        print(f"   {'pass@1=%.3f' % acc if acc is not None else 'ungraded'}  "
              f"truncated={sum(r['truncated'] for r in recs) / len(recs):.2%}  "
              f"tokens={meta['total_gen_tokens'] / 1e6:.2f}M  {gen_s / 60:.1f} min -> {path}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/analyze.py
"""Load gen.py results, compute pass@k, arithmetic error rates, and plot.

Usage in a notebook:
    import analyze as an
    df = an.load_results("results/math-1.5b/math500")
    an.summary(df)                 # one row per (tag, condition)
    an.calls_table(df)             # tool-call behaviour: how many called, how often, retries after a denial
    an.plot(df, ["tool_rl prompt_off", "text_rl prompt_off", "base prompt_off"])   # one panel
    an.plot(df, {"base": ["base prompt_off", "base tool_on"], "tool_rl": [...]})    # one panel per key
    an.plot_passk(df)              # older view: one panel per condition, models as lines
    an.verdict(df)                 # atrophy / internalization / overlap readout
"""
import json
import math
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

ROLE_ORDER = ["base", "text_rl", "tool_rl"]
# dataviz reference palette, categorical slots 1-3 (CVD-validated): blue, orange, aqua
ROLE_COLOR = {"base": "#2a78d6", "tool_rl": "#eb6834", "text_rl": "#1baf7a"}
COND_ORDER = ["prompt_off", "env_denied", "tool_on", "plain_cot"]
COND_STYLE = {"prompt_off": "-", "env_denied": "--", "tool_on": ":", "plain_cot": "-."}
COND_MARK = {"prompt_off": "o", "env_denied": "s", "tool_on": "^", "plain_cot": "D"}
# conditions get hues of their own (reference palette slots 7, 8, 4, 6, validated as a set) so a
# per-model panel never reuses a model color
COND_COLOR = {"prompt_off": "#6250d6", "env_denied": "#e34948", "tool_on": "#eda100", "plain_cot": "#008300"}
ROLE_STYLE = {"base": "-", "text_rl": "--", "tool_rl": ":"}
GRAY = "#52514e"
# default arms of the headline figure: tool-RL with the tool off against the control and the base
MAIN_ARMS = [("tool_rl", "prompt_off"),
             ("text_rl", "prompt_off"), ("text_rl", "tool_on"),
             ("base", "prompt_off"), ("base", "tool_on")]


def load_results(results_dir, allow_ungraded=False) -> pd.DataFrame:
    """Every *.jsonl in the folder as one DataFrame. Raises if any record is still ungraded
    (gen.py --no-grade) unless allow_ungraded, since pass@k would silently be wrong."""
    paths = sorted(Path(results_dir).glob("*.jsonl"))
    if not paths:
        raise FileNotFoundError(f"no .jsonl under {results_dir}")
    frames = []
    for p in paths:
        rows = [json.loads(line) for line in open(p)]
        f = pd.DataFrame(rows)
        f["file"] = p.name
        frames.append(f)
    df = pd.concat(frames, ignore_index=True)
    df["arm"] = df["tag"] + " / " + df["condition"]
    n_un = int(df["correct"].isna().sum()) if "correct" in df else len(df)
    if n_un and not allow_ungraded:
        raise ValueError(f"{n_un} of {len(df)} records in {results_dir} are ungraded: "
                         f"run  python src/grade.py {results_dir}  first")
    if not n_un:
        df["correct"] = df["correct"].astype(bool)
    return df


def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased estimator from Chen et al. 2021."""
    if k > n:
        return float("nan")
    if n - c < k:
        return 1.0
    return 1.0 - math.exp(math.lgamma(n - c + 1) - math.lgamma(n - c - k + 1)
                          - math.lgamma(n + 1) + math.lgamma(n - k + 1))


def ks_for(n: int):
    ks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    return [k for k in ks if k <= n]


def per_problem(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["tag", "condition", "problem_id"])["correct"]
    out = g.agg(n="count", c="sum").reset_index()
    return out


def passk_table(df: pd.DataFrame) -> pd.DataFrame:
    """Rows: (tag, condition); columns: pass@k means over problems."""
    pp = per_problem(df)
    rows = []
    for (tag, cond), grp in pp.groupby(["tag", "condition"]):
        n = int(grp["n"].min())
        row = dict(tag=tag, condition=cond, n_problems=len(grp), n_samples=n)
        for k in ks_for(n):
            row[f"pass@{k}"] = float(np.mean([pass_at_k(int(r.n), int(r.c), k) for r in grp.itertuples()]))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["condition", "tag"]).reset_index(drop=True)


def bootstrap_passk(df: pd.DataFrame, tag: str, cond: str, k: int, iters: int = 1000, seed: int = 0):
    """Percentile bootstrap over problems. Returns (mean, lo, hi)."""
    pp = per_problem(df)
    grp = pp[(pp.tag == tag) & (pp.condition == cond)]
    vals = np.array([pass_at_k(int(r.n), int(r.c), k) for r in grp.itertuples()])
    if len(vals) == 0:
        return (float("nan"),) * 3
    rng = np.random.default_rng(seed)
    boots = rng.choice(vals, size=(iters, len(vals)), replace=True).mean(axis=1)
    return float(vals.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def paired_diff(df: pd.DataFrame, a: tuple, b: tuple, k: int, iters: int = 1000, seed: int = 0):
    """Bootstrap CI of pass@k(a) - pass@k(b), paired by problem. a, b = (tag, condition)."""
    pp = per_problem(df)
    A = pp[(pp.tag == a[0]) & (pp.condition == a[1])].set_index("problem_id")
    B = pp[(pp.tag == b[0]) & (pp.condition == b[1])].set_index("problem_id")
    ids = A.index.intersection(B.index)
    if len(ids) == 0:
        return (float("nan"),) * 3
    da = np.array([pass_at_k(int(A.loc[i, "n"]), int(A.loc[i, "c"]), k) for i in ids])
    db = np.array([pass_at_k(int(B.loc[i, "n"]), int(B.loc[i, "c"]), k) for i in ids])
    d = da - db
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, size=(iters, len(d)), replace=True).mean(axis=1)
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summary(df: pd.DataFrame) -> pd.DataFrame:
    """Headline table: accuracy, pass@kmax, failure modes, arithmetic error rate."""
    pk = passk_table(df)
    rows = []
    for (tag, cond), g in df.groupby(["tag", "condition"]):
        n = int(g.groupby("problem_id").size().min())
        kmax = ks_for(n)[-1]
        pkrow = pk[(pk.tag == tag) & (pk.condition == cond)].iloc[0]
        arith_n = int(g["arith_n"].sum())
        rows.append(dict(
            tag=tag, condition=cond, template=g["template"].iloc[0],
            problems=g["problem_id"].nunique(), samples=n,
            **{"pass@1": round(pkrow["pass@1"], 3), f"pass@{kmax}": round(pkrow[f"pass@{kmax}"], 3)},
            trunc_pct=round(100 * g["truncated"].mean(), 1),
            no_boxed_pct=round(100 * (~g["boxed_present"]).mean(), 1),
            code_pct=round(100 * g["code_emitted"].mean(), 1),
            fake_out_pct=round(100 * (g["fake_outputs"] > 0).mean(), 1),
            called_pct=round(100 * (g["n_calls"] > 0).mean(), 1),
            calls_mean=round(g["n_calls"].mean(), 2), denied_mean=round(g["n_denied"].mean(), 2),
            exec_err_mean=round(g["n_exec_err"].mean(), 2),
            arith_err_pct=round(100 * g["arith_wrong"].sum() / arith_n, 2) if arith_n else float("nan"),
            arith_checked=arith_n, arith_coverage_pct=round(100 * (g["arith_n"] > 0).mean(), 1),
            tokens_mean=round(g["gen_tokens"].mean()),
        ))
    out = pd.DataFrame(rows)
    out["_r"] = out["tag"].map({t: i for i, t in enumerate(ROLE_ORDER)}).fillna(9)
    return out.sort_values(["condition", "_r"]).drop(columns="_r").reset_index(drop=True)


def _plot_lines(df, pk, ax, ci, iters, label_cond=True):
    for _, r in pk.iterrows():
        tag, cond = r["tag"], r["condition"]
        ks = ks_for(int(r["n_samples"]))
        ys = [r[f"pass@{k}"] for k in ks]
        color = ROLE_COLOR.get(tag, "#52514e")
        ax.plot(ks, ys, COND_STYLE.get(cond, "-"), color=color, lw=2,
                marker=COND_MARK.get(cond, "o"), ms=6,
                label=f"{tag} / {cond}" if label_cond else tag)
        if ci:
            lo, hi = [], []
            for k in ks:
                _, l, h = bootstrap_passk(df, tag, cond, k, iters=iters)
                lo.append(l); hi.append(h)
            ax.fill_between(ks, lo, hi, color=color, alpha=0.12, lw=0)
    kmax = ks_for(int(pk["n_samples"].max()))
    ax.set_xscale("log", base=2)
    ax.set_xticks(kmax)
    ax.set_xticklabels([str(k) for k in kmax])
    ax.set_xlabel("k (samples)")
    ax.set_ylim(0, 1)
    ax.grid(True, color="#e6e5e1", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower right")


def plot_passk(df: pd.DataFrame, title=None, ci=True, iters=300, facet=True):
    """One panel per condition (three models per panel), or one panel with every arm."""
    import matplotlib.pyplot as plt
    pk = passk_table(df)
    conds = [c for c in COND_ORDER if c in set(pk.condition)] + \
            sorted(set(pk.condition) - set(COND_ORDER))
    if not facet or len(conds) == 1:
        fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
        _plot_lines(df, pk, ax, ci, iters)
        ax.set_ylabel("pass@k")
        ax.set_title(title or f"pass@k, {df['problem_id'].nunique()} problems", loc="left")
        return fig
    fig, axes = plt.subplots(1, len(conds), figsize=(4.6 * len(conds), 4.2), dpi=120, sharey=True)
    for ax, cond in zip(axes, conds):
        sub = pk[pk.condition == cond].copy()
        sub["_r"] = sub["tag"].map({t: i for i, t in enumerate(ROLE_ORDER)}).fillna(9)
        _plot_lines(df[df.condition == cond], sub.sort_values("_r"), ax, ci, iters, label_cond=False)
        ax.set_title(cond, loc="left")
    axes[0].set_ylabel("pass@k")
    fig.suptitle(title or f"pass@k, {df['problem_id'].nunique()} problems", x=0.01, ha="left")
    fig.tight_layout()
    return fig


def calls_table(df: pd.DataFrame) -> pd.DataFrame:
    """Tool-call behaviour per arm.

    called_pct          % of samples with at least one intercepted python block
    calls_total         all calls in the arm (sum over samples)
    calls_per_caller    mean calls among the samples that called at all
    denied_total        calls answered with a denial message
    retry_after_denial  mean further calls after the first denial, among samples denied at least once.
                        Once a call is denied every later call in that sample is denied too (prompt_off
                        and env_denied deny all, tool_on denies everything past the budget), so this is
                        n_denied - 1.
    retry_pct           % of denied samples that called again anyway
    calls=k             number of samples that made exactly k calls (5 and more pooled)
    """
    rows = []
    for (tag, cond), g in df.groupby(["tag", "condition"]):
        callers = g[g["n_calls"] > 0]
        denied = g[g["n_denied"] > 0]
        row = dict(tag=tag, condition=cond, samples=len(g),
                   called_pct=round(100 * (g["n_calls"] > 0).mean(), 1),
                   calls_total=int(g["n_calls"].sum()),
                   calls_per_caller=round(float(callers["n_calls"].mean()), 2) if len(callers) else float("nan"),
                   denied_total=int(g["n_denied"].sum()),
                   retry_after_denial=round(float((denied["n_denied"] - 1).mean()), 2) if len(denied) else float("nan"),
                   retry_pct=round(100 * float((denied["n_denied"] >= 2).mean()), 1) if len(denied) else float("nan"))
        hist = g["n_calls"].clip(upper=5).value_counts()
        for k in range(6):
            row["calls>=5" if k == 5 else f"calls={k}"] = int(hist.get(k, 0))
        rows.append(row)
    out = pd.DataFrame(rows)
    out["_r"] = out["tag"].map({t: i for i, t in enumerate(ROLE_ORDER)}).fillna(9)
    out["_c"] = out["condition"].map({c: i for i, c in enumerate(COND_ORDER)}).fillna(9)
    return out.sort_values(["_c", "_r"]).drop(columns=["_r", "_c"]).reset_index(drop=True)


def file_stamp() -> str:
    """Local date and time for a figure filename, e.g. 2026-10-14_1530."""
    return datetime.now().strftime("%Y-%m-%d_%H%M")


def provenance(results_dir) -> str:
    """One-line figure caption: when the arms in results_dir were generated (each arm's meta
    finished_at, else the file's modification time), how many there are, and when the plot was made."""
    paths = sorted(Path(results_dir).glob("*.jsonl")) if Path(results_dir).is_dir() else []
    times = []
    for p in paths:
        t = None
        m = p.with_suffix(".meta.json")
        if m.exists():
            try:
                t = json.loads(m.read_text()).get("finished_at")
            except (json.JSONDecodeError, OSError):
                t = None
        times.append((datetime.fromisoformat(t) if t else datetime.fromtimestamp(p.stat().st_mtime)).astimezone())
    if not times:
        return ""
    lo, hi = min(times), max(times)
    span = lo.strftime("%Y-%m-%d %H:%M")
    if hi - lo >= timedelta(minutes=1):
        span += " to " + hi.strftime("%Y-%m-%d %H:%M")
    return f"{len(paths)} arms generated {span}; plotted {datetime.now():%Y-%m-%d %H:%M}"


def _format_axis(ax, n_max):
    ks = ks_for(int(n_max))
    ax.set_xscale("log", base=2)
    ax.set_xticks(ks)
    ax.set_xticklabels([str(k) for k in ks])
    ax.set_xlabel("k (samples)")
    ax.set_ylim(0, 1)
    ax.grid(True, color="#e6e5e1", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def _draw_arm(ax, df, pk, tag, cond, color, style, label, ci, iters):
    """One pass@k curve (plus bootstrap band) for arm (tag, cond). Returns False if absent."""
    row = pk[(pk.tag == tag) & (pk.condition == cond)]
    if row.empty:
        return False
    r = row.iloc[0]
    ks = ks_for(int(r["n_samples"]))
    ax.plot(ks, [r[f"pass@{k}"] for k in ks], style, color=color, lw=2,
            marker=COND_MARK.get(cond, "o"), ms=6, label=label)
    if ci:
        lo, hi = zip(*[bootstrap_passk(df, tag, cond, k, iters=iters)[1:] for k in ks])
        ax.fill_between(ks, lo, hi, color=color, alpha=0.12, lw=0)
    return True


def _arm(line):
    """'tool_rl prompt_off', 'tool_rl/prompt_off' or ('tool_rl', 'prompt_off') -> ('tool_rl', 'prompt_off')."""
    if isinstance(line, str):
        parts = line.replace("/", " ").split()
        if len(parts) != 2:
            raise ValueError(f"line must be 'model condition', got {line!r}")
        return parts[0], parts[1]
    tag, cond = line
    return tag, cond


def plot(df: pd.DataFrame, lines, title=None, color="model", ci=True, iters=300, save=None, caption=None):
    """Draw pass@k curves.

    lines  a list of "model condition" strings for one panel, or a dict {panel title: [lines]} for
           several panels sharing the y axis. Models: base, text_rl, tool_rl. Conditions: prompt_off,
           env_denied, tool_on, plain_cot. "tool_rl/prompt_off" and ("tool_rl", "prompt_off") also work.
    color  "model" or "condition": what the color encodes. The other one sets the line style, but only
           when it varies within a panel.
    ci     shade a 95% bootstrap band over problems.
    save   path of a PNG to write (use file_stamp() in the name to date it).
    caption small gray line at the bottom, e.g. provenance(results_dir).
    Lines whose arm is not in df are listed and skipped.
    """
    import matplotlib.pyplot as plt
    panels = lines if isinstance(lines, dict) else {None: lines}
    pk = passk_table(df)
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(7 if n == 1 else 4.6 * n, 4.4), dpi=120, sharey=True, squeeze=False)
    missing = []
    for ax, (name, arms) in zip(axes[0], panels.items()):
        arms = [_arm(a) for a in arms]
        vary_tag, vary_cond = len({t for t, _ in arms}) > 1, len({c for _, c in arms}) > 1
        for tag, cond in arms:
            if color == "condition":
                col = COND_COLOR.get(cond, GRAY)
                style = ROLE_STYLE.get(tag, "-") if vary_tag else "-"
            else:
                col = ROLE_COLOR.get(tag, GRAY)
                style = COND_STYLE.get(cond, "-") if vary_cond else "-"
            if not _draw_arm(ax, df, pk, tag, cond, col, style, f"{tag} {cond}", ci, iters):
                missing.append(f"{tag} {cond}")
        _format_axis(ax, pk["n_samples"].max())
        if name:
            ax.set_title(name, loc="left")
        if ax.get_lines():
            ax.legend(frameon=False, fontsize=8, loc="lower right")
    axes[0][0].set_ylabel("pass@k")
    if title:
        if n == 1:
            axes[0][0].set_title(title, loc="left")
        else:
            fig.suptitle(title, x=0.01, ha="left")
    if missing:
        print("not in results, skipped:", ", ".join(missing))
    if caption:
        fig.text(0.01, 0.004, caption, fontsize=7, color=GRAY, ha="left", va="bottom")
        fig.tight_layout(rect=(0, 0.035, 1, 1))
    else:
        fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def plot_passk_by_model(df: pd.DataFrame, title=None, ci=True, iters=300):
    """One panel per model, its conditions as lines. Same as plot(df, {model: [...]}, color="condition")."""
    pk = passk_table(df)
    tags = [t for t in ROLE_ORDER if t in set(pk.tag)] + sorted(set(pk.tag) - set(ROLE_ORDER))
    panels = {t: [f"{t} {c}" for c in COND_ORDER if c in set(pk[pk.tag == t].condition)] for t in tags}
    return plot(df, panels, title=title or f"pass@k within each model, {df['problem_id'].nunique()} problems",
                color="condition", ci=ci, iters=iters)


def plot_passk_main(df: pd.DataFrame, arms=MAIN_ARMS, title=None, ci=True, iters=300):
    """The headline arms on one axis. Same as plot(df, [...])."""
    return plot(df, list(arms), title=title or f"pass@k, {df['problem_id'].nunique()} problems", ci=ci, iters=iters)


def verdict(df: pd.DataFrame, cond: str = "env_denied", iters: int = 1000):
    """Print the three-way readout from the plan for one condition (same prompt, all models)."""
    pk = passk_table(df)
    sub = pk[pk.condition == cond]
    tags = set(sub.tag)
    need = {"base", "text_rl", "tool_rl"}
    if not need <= tags:
        print(f"[{cond}] need base, text_rl, tool_rl; have {sorted(tags)}")
        return
    n = int(sub.n_samples.min())
    kmax = ks_for(n)[-1]
    print(f"[{cond}] {sub.n_problems.min()} problems, n={n}, kmax={kmax}")
    for k in (1, kmax):
        for a, b in ((("tool_rl", cond), ("text_rl", cond)), (("tool_rl", cond), ("base", cond)),
                     (("text_rl", cond), ("base", cond))):
            m, lo, hi = paired_diff(df, a, b, k, iters=iters)
            sig = "" if lo <= 0 <= hi else "  *"
            print(f"  pass@{k:<4} {a[0]:>7} - {b[0]:<7} = {m:+.3f}  [{lo:+.3f}, {hi:+.3f}]{sig}")
    g = df[df.condition == cond].groupby("tag")
    er = (g["arith_wrong"].sum() / g["arith_n"].sum() * 100).round(2)
    print("  arithmetic error rate (%):", er.to_dict())
    d1 = paired_diff(df, ("tool_rl", cond), ("text_rl", cond), 1, iters=iters)
    dk = paired_diff(df, ("tool_rl", cond), ("text_rl", cond), kmax, iters=iters)
    if dk[2] < 0:
        print("  -> tool_rl below text_rl at high k: ATROPHY signature")
    elif d1[1] > 0 and dk[1] > 0:
        print("  -> tool_rl above text_rl at pass@1 and high k: INTERNALIZATION signature")
    elif d1[1] <= 0 <= d1[2] and dk[1] <= 0 <= dk[2]:
        print("  -> tool_rl and text_rl overlap at every k: tool added nothing beyond RL (kill criterion)")
    else:
        print("  -> mixed: pass@1 gain without a high-k deficit, or vice versa; look at the curve")

## 4. Configure

Presets name a (base, text-RL, tool-RL) trio from one base model. Each model also
carries its tool prompt: ToRL's own (paper Fig. 3) for ToRL, the Qwen2.5-Math TIR
prompt (same instruction sentence, chat format) for base and Oat-Zero, CIR's Fig. 5
text for the CIR checkpoint. Set every `native` to the same key if you want one
identical tool prompt for all three.

| Preset | base | text-RL (control) | tool-RL |
|---|---|---|---|
| `math-1.5b` | Qwen/Qwen2.5-Math-1.5B | sail/Qwen2.5-Math-1.5B-Oat-Zero | GAIR/ToRL-1.5B |
| `math-7b` | Qwen/Qwen2.5-Math-7B | sail/Qwen2.5-Math-7B-Oat-Zero | RUC-AIBOX/Qwen-Math-7B-CIR (the Fig. 6 model) |
| `math-7b-torl` | Qwen/Qwen2.5-Math-7B | sail/Qwen2.5-Math-7B-Oat-Zero | GAIR/ToRL-7B |
| `general-7b-zerotir` | Qwen/Qwen2.5-7B | hkust-nlp/Qwen-2.5-7B-SimpleRL-Zoo | htxu91/zero-tir-7b-550step (tool prompt unverified) |

Context is capped at 4096 tokens for the Qwen2.5-Math trios because that is the
base model's and Oat-Zero's native context (CIR evaluated at 16k by extrapolating;
`ALLOW_LONG=True` reproduces that). Truncation rate is reported either way.

`PLAIN_COT=True` adds the fourth, non-intercepting condition for a tie to CIR's
numbers. `DENIAL_TEXT` overrides the message injected under `prompt_off` and
`env_denied`.

In [ ]:
PRESETS = {
    "math-1.5b": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-1.5B",            native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-1.5B-Oat-Zero",   native="qwen_math_tir"),
        "tool_rl": dict(model="GAIR/ToRL-1.5B",                     native="torl"),
    }),
    "math-7b": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-7B",              native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-7B-Oat-Zero",     native="qwen_math_tir"),
        "tool_rl": dict(model="RUC-AIBOX/Qwen-Math-7B-CIR",        native="cir"),
    }),
    "math-7b-torl": dict(max_model_len=4096, models={
        "base":    dict(model="Qwen/Qwen2.5-Math-7B",              native="qwen_math_tir"),
        "text_rl": dict(model="sail/Qwen2.5-Math-7B-Oat-Zero",     native="qwen_math_tir"),
        "tool_rl": dict(model="GAIR/ToRL-7B",                       native="torl"),
    }),
    "general-7b-zerotir": dict(max_model_len=8192, models={
        "base":    dict(model="Qwen/Qwen2.5-7B",                   native="qwen_math_tir"),
        "text_rl": dict(model="hkust-nlp/Qwen-2.5-7B-SimpleRL-Zoo", native="qwen_math_tir"),
        "tool_rl": dict(model="htxu91/zero-tir-7b-550step",         native="qwen_math_tir"),
    }),
}

PRESET = "math-1.5b"       # start here; "math-7b" once the 1.5B run is clean
DATASET = "math500"        # math500 | aime24 | aime25
SMOKE = True               # True: 20 problems x 4 samples x 1024 tokens, ~10 min. False: the real run.

N_PROBLEMS = 20 if SMOKE else 0        # 0 = all 500 (AIME sets have 30)
N_SAMPLES = 4 if SMOKE else 16         # raise to 32 or 64 for a sharper high-k tail
MAX_TOKENS = 1024 if SMOKE else 3000
TEMPERATURE, TOP_P, SEED = 0.6, 0.95, 0   # CIR Fig. 6 settings
MAX_CALLS = 4
ALLOW_LONG = False         # True + MAX_MODEL_LEN=16384 mimics CIR's 16k budget on 4k-native models
MAX_MODEL_LEN = PRESETS[PRESET]["max_model_len"]
SHARED_TEMPLATE = "qwen_math_nocode"   # prompt_off prompt, identical for every model
PLAIN_COT = False          # True adds the non-intercepting plain_cot arm (CIR's condition)
DENIAL_TEXT = None         # None = "No interpreter is available; the code was not executed. ..."
GPU_MEM = 0.90
CHUNK = 100                # problems per checkpoint; a disconnect loses at most this much work
OVERWRITE = False          # True regenerates arms that are already on Drive
GRADE_ON_COLAB = False     # False: Colab only generates; grade + analyze on your laptop (section 6)

CONDITIONS = ["prompt_off", "env_denied", "tool_on"] + (["plain_cot"] if PLAIN_COT else [])
RUN_NAME = f"{PRESET}{'-smoke' if SMOKE else ''}"
OUT = f"{RESULTS}/{RUN_NAME}"
print(RUN_NAME, "->", OUT)

## 5. Run

One cell per checkpoint, below. Each cell first looks at what Drive already holds
for that model:

| On Drive | What the cell does |
|---|---|
| `.meta.json` for the arm | finished: loaded as is, nothing generated |
| `.jsonl.partial` only | interrupted: resumes from the last checkpointed chunk of `CHUNK` problems |
| nothing | generated from scratch |

So after a disconnect, reconnect, re-run cells 1 to 4 (install, Drive, code,
config), then re-run the model cell that was going. Finished arms and finished
chunks are never redone unless `OVERWRITE = True`. The three cells can run in any
order, one at a time (one GPU). Each one ends with a quick pass@1 table read back
from Drive. The first run of each model spends a few minutes downloading weights.

`samples` in that table is rows, which is problems x `N_SAMPLES`. If it is a
multiple of what you expect, a file from an older run with a different prompt
template sits in the same folder and loads as the same arm. `status` flags this;
`move_stale()` moves such files into `old/`.

In [ ]:
import json, os, shutil, subprocess, sys, time, importlib
from pathlib import Path
sys.path.insert(0, "src")


def arm_files(tag):
    spec = PRESETS[PRESET]["models"][tag]
    out = Path(OUT) / DATASET
    rows = []
    for cond in CONDITIONS:
        tmpl = SHARED_TEMPLATE if cond == "prompt_off" else ("qwen_math_cot" if cond == "plain_cot" else spec["native"])
        stem = out / f"{tag}__{cond}__{tmpl}"
        rows.append((cond, stem.with_suffix(".jsonl"), stem.with_suffix(".meta.json"), Path(str(stem) + ".jsonl.partial")))
    return rows


def partial_problems(partial):
    pids = set()
    for line in open(partial):
        try:
            pids.add(json.loads(line)["problem_id"])
        except (json.JSONDecodeError, KeyError):
            pass  # torn last line
    return len(pids)


DATA_SUFFIXES = (".jsonl", ".meta.json", ".jsonl.partial", ".jsonl.tmp")


def expected_stems():
    return {jsonl.stem for tag in PRESETS[PRESET]["models"] for _, jsonl, _, _ in arm_files(tag)}


def stale_files():
    # Data files in the run folder that the current config does not produce (an older template or
    # model), plus arms written by an older gen.py (no "chunk" key in meta). They inflate the
    # tables (two files load as one arm) and block regeneration (gen.py sees the arm as done).
    out = Path(OUT) / DATASET
    exp = expected_stems()
    stale = []
    for f in sorted(out.iterdir()) if out.is_dir() else []:
        if not f.is_file() or not f.name.endswith(DATA_SUFFIXES):
            continue
        stem = f.name.split(".")[0]
        if stem not in exp:
            stale.append((f, "not an arm of the current config"))
        elif f.name.endswith(".meta.json") and "chunk" not in json.load(open(f)):
            stale.append((f, "written by an older notebook version"))
            stale.append((out / f"{stem}.jsonl", "written by an older notebook version"))
    return stale


def move_stale():
    # Quarantine stale files into <run>/<dataset>/old/ (nothing is deleted).
    out = Path(OUT) / DATASET
    stale = stale_files()
    if not stale:
        print("nothing stale in", out); return
    (out / "old").mkdir(exist_ok=True)
    for f, why in stale:
        if f.exists():
            shutil.move(str(f), str(out / "old" / f.name))
            print(f"moved {f.name} -> old/   ({why})")


def load_run(require_graded=False):
    # Results of the current config only; files from other runs in the folder are reported and ignored.
    # require_graded raises with instructions if rows still wait for src/grade.py.
    import analyze as an
    importlib.reload(an)
    df = an.load_results(f"{OUT}/{DATASET}", allow_ungraded=not require_graded)
    exp = {f"{stem}.jsonl" for stem in expected_stems()}
    stale = {f.name for f, _ in stale_files()}
    stray = sorted((set(df.file) - exp) | (set(df.file) & stale))
    if stray:
        print("ignoring", len(stray), "file(s) from another run or notebook version:", ", ".join(stray))
        print("run move_stale() to quarantine them")
    return df[~df.file.isin(stray)].copy()


def status(tag):
    # What Drive holds for this model. Returns how many arms still need generation.
    missing = 0
    for cond, jsonl, meta, partial in arm_files(tag):
        if meta.exists():
            m = json.load(open(meta))
            acc = "ungraded" if m.get("pass1") is None else f"pass@1={m['pass1']:.3f}"
            note = f"done     {acc}  n={m['n']}  problems={m['n_problems']}  {m['gen_seconds'] / 60:.0f} min"
            if m["n"] != N_SAMPLES or (N_PROBLEMS and m["n_problems"] != N_PROBLEMS):
                note += "   <- written with a different N_SAMPLES/N_PROBLEMS; set OVERWRITE=True or change RUN_NAME"
            if "chunk" not in m:
                note += "   <- older notebook version; run move_stale(), then re-run this cell"
        elif partial.exists():
            note = f"partial  {partial_problems(partial)} problems checkpointed, will resume"
            missing += 1
        else:
            note = "missing"
            missing += 1
        print(f"  {tag:8s} {cond:11s} {note}")
    n_stale = len({f for f, _ in stale_files()})
    if n_stale:
        print(f"  ! {n_stale} stale file(s) from another run in this folder: run move_stale() first")
    return missing


def show(tag):
    import analyze as an
    try:
        df = load_run()
    except FileNotFoundError:
        print("no results yet"); return
    d = df[df.tag == tag]
    if d.empty:
        print(f"no finished arms for {tag} yet"); return
    graded = d["correct"].notna().all()
    if not graded:
        print("rows are ungraded: pass@1 appears after  python src/grade.py <this folder>  on your laptop")
    aggs = dict(samples=("correct", "size"), problems=("problem_id", "nunique"), trunc=("truncated", "mean"),
                called=("n_calls", lambda s: (s > 0).mean()), calls_total=("n_calls", "sum"),
                calls=("n_calls", "mean"), denied=("n_denied", "mean"))
    if graded:
        d = d.assign(correct=d["correct"].astype(float))
        aggs = dict(samples=aggs["samples"], problems=aggs["problems"], pass1=("correct", "mean"),
                    **{k: v for k, v in aggs.items() if k not in ("samples", "problems")})
    tbl = d.groupby("condition").agg(**aggs).round(3)
    tbl = tbl.reindex([c for c in an.COND_ORDER if c in tbl.index])
    try:
        display(tbl)
    except NameError:
        print(tbl.to_string())


def stream(cmd):
    # Relay the subprocess's stdout+stderr through the kernel: a line per tqdm refresh instead of
    # a carriage-return bar, so progress shows even when Colab does not mirror child output.
    # The child runs in its own process group; the stop button kills the whole group (gen.py and
    # vLLM's engine-core process), so nothing is left holding the GPU.
    import signal
    env = dict(os.environ, TQDM_MININTERVAL="20", PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                            start_new_session=True)
    try:
        while True:
            chunk = proc.stdout.read1(65536)
            if not chunk:
                break
            print(chunk.decode("utf-8", "replace").replace("\r", "\n"), end="", flush=True)
        return proc.wait()
    except BaseException:
        for sig, wait in ((signal.SIGTERM, 15), (signal.SIGKILL, 5)):
            try:
                os.killpg(proc.pid, sig)
                proc.wait(timeout=wait)
                break
            except subprocess.TimeoutExpired:
                continue
            except ProcessLookupError:
                break
        print(f"\n[interrupted] killed pid {proc.pid} and its children; finished arms and chunks are kept",
              flush=True)
        raise


def run_model(tag):
    spec = PRESETS[PRESET]["models"][tag]
    print(f"[{tag}] {spec['model']}  ->  {OUT}/{DATASET}")
    if status(tag) == 0 and not OVERWRITE:
        print(f"[{tag}] every arm is already on Drive, loading instead of regenerating")
    elif not shutil.which("nvidia-smi"):
        print(f"[{tag}] arms are missing but this runtime has NO GPU, nothing generated.\n"
              "  Colab: Runtime > Change runtime type > GPU. VS Code: reconnect choosing a GPU runtime.\n"
              "  Then run again from cell 1 (a runtime change resets the VM).")
    else:
        cmd = [sys.executable, "src/gen.py",
               "--model", spec["model"], "--tag", tag,
               "--conditions", ",".join(CONDITIONS),
               "--shared-template", SHARED_TEMPLATE, "--native-template", spec["native"],
               "--dataset", DATASET, "--n-problems", str(N_PROBLEMS), "--n", str(N_SAMPLES),
               "--max-tokens", str(MAX_TOKENS), "--max-model-len", str(MAX_MODEL_LEN),
               "--temperature", str(TEMPERATURE), "--top-p", str(TOP_P), "--seed", str(SEED),
               "--max-calls", str(MAX_CALLS), "--gpu-mem", str(GPU_MEM), "--chunk", str(CHUNK), "--out", OUT]
        if ALLOW_LONG:
            cmd.append("--allow-long")
        if OVERWRITE:
            cmd.append("--overwrite")
        if not GRADE_ON_COLAB:
            cmd.append("--no-grade")
        if DENIAL_TEXT:
            cmd += ["--denial-text", DENIAL_TEXT]
        print("\n$", " ".join(cmd), flush=True)
        t0 = time.time()
        rc = stream(cmd)
        print(f"\n[{tag}] exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
        if rc != 0:
            print("fix the error above and re-run this cell; finished arms and chunks are kept")
        status(tag)
    show(tag)

In [ ]:
run_model("base")

In [ ]:
run_model("text_rl")

In [ ]:
run_model("tool_rl")

## 6. Analyze

With `GRADE_ON_COLAB = False` (the default) the rows on Drive are ungraded and the
cells below stop with a message. Grade and analyze on your laptop instead:

```
cd ~/projects/239_project
uv sync                                                        # once
uv run src/report.py --run math-1.5b-smoke --dataset math500
```

`report.py --run` finds the run under Google Drive for desktop (any
`239_project/results` up to five levels under My Drive), grades the ungraded rows
in place first (`src/grade.py`, skips rows already
graded), then prints the tables and saves the two figures next to the data. Pass a
folder instead of `--run` for an unzipped download from the last cell. The same functions are available in a local notebook via
`import analyze as an` (`an.load_results`, `an.summary`, `an.plot`).

With `GRADE_ON_COLAB = True` everything below runs here.

`summary` is the headline table, one row per arm. `trunc_pct`, `no_boxed_pct`,
`code_pct` and `fake_out_pct` are the failure modes that would make a pass@k gap
uninterpretable, so read them before the curves. Under `prompt_off`, `called_pct`
is the share of samples that wrote code after being told not to; `calls_table`
breaks tool use down further (calls per caller, retries after a denial). `arith_err_pct` is the direct skill metric
(exact-rational check of every `a op b = c` in the prose, code blocks excluded);
`arith_coverage_pct` says how many samples contained anything to check.

Figures come from `an.plot(df, lines)`, where each line is a `"model condition"`
string. The two cells below draw the per-model view (what turning the tool off
costs each checkpoint) and the headline view (tool-RL with the tool off against the
control and the base). Edit the strings to draw anything else.

In [ ]:
import sys, importlib
sys.path.insert(0, "src")
import analyze as an
importlib.reload(an)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

if GRADE_ON_COLAB:
    df = load_run(require_graded=True)   # current config's files only; strays from older runs are listed and skipped
    print(len(df), "samples,", df.problem_id.nunique(), "problems,", df.arm.nunique(), "arms")
    STAMP, CAPTION = an.file_stamp(), an.provenance(f"{OUT}/{DATASET}")   # figures are dated and captioned
    print(CAPTION)
    display(an.summary(df))
else:
    df = None
    print("Generation done. Rows are ungraded here (GRADE_ON_COLAB = False). On your laptop:\n"
          f"  cd ~/projects/239_project && uv run src/report.py --run {RUN_NAME} --dataset {DATASET}")

In [ ]:
if df is not None:
    display(an.passk_table(df).round(3))

In [ ]:
# Tool-call behaviour. called_pct: % of samples that wrote a python block (under prompt_off: after being
# told not to). calls_per_caller: calls among those. retry_after_denial: further calls after the first
# denial (every call after a denial is denied too, so this is n_denied - 1). calls=k: samples by call count.
if df is not None:
    display(an.calls_table(df))

In [ ]:
# an.plot draws pass@k curves. Each line is "model condition".
#   models:     base, text_rl, tool_rl
#   conditions: prompt_off, env_denied, tool_on
# A list makes one panel. A dict {panel title: [lines]} makes one panel per key.
# color="model" or "condition" picks what the color means; the other one sets the line style.
# save=... writes a PNG next to the results (dated with STAMP); caption= puts the data's generation
# dates under the plot. Lines not in the results are skipped with a note.

# Figure 1: within each model, what does turning the tool off cost?
if df is not None:
    an.plot(df, {
        "base":    ["base prompt_off",    "base env_denied",    "base tool_on"],
        "text_rl": ["text_rl prompt_off", "text_rl env_denied", "text_rl tool_on"],
        "tool_rl": ["tool_rl prompt_off", "tool_rl env_denied", "tool_rl tool_on"],
    }, color="condition", title=f"{RUN_NAME}: pass@k within each model",
       save=f"{OUT}/{DATASET}/passk_by_model_{STAMP}.png", caption=CAPTION);

In [ ]:
# Figure 2: tool-RL with the tool off, against the text-RL control and the base model
if df is not None:
    an.plot(df, [
        "tool_rl prompt_off",
        "text_rl prompt_off", "text_rl tool_on",
        "base prompt_off",    "base tool_on",
    ], title=f"{RUN_NAME}: tool-RL with the tool off vs control and base",
       save=f"{OUT}/{DATASET}/passk_main_{STAMP}.png", caption=CAPTION);

In [ ]:
if df is not None:
    for cond in an.COND_ORDER:
        if cond in set(df.condition):
            an.verdict(df, cond)
            print()

## 7. Reading the result

- **Read within a column.** Each condition puts all three models under the same
  prompt and the same harness, so `verdict` compares them per condition.
  `env_denied` is the headline: every model asked for code, nobody gets to run it.
- **prompt_off tests compliance too.** `calls_mean` on those rows is how often a
  model wrote code after being told not to. A tool-RL model that cannot stop is
  itself a finding.
- **Compare to text_rl, not just base.** RL alone sharpens pass@1 and can lower
  high-k pass@k (the sharpening confound). The question is whether tool-RL sits
  below text-RL.
- **Check failure modes first.** If an arm has 30% truncation or never boxes an
  answer, its curve measures format collapse.
- **Arithmetic error rate** is the skill the interpreter replaces. Tool-RL above
  text-RL here, at the same accuracy, is atrophy of exactly that skill.

Then: flip `SMOKE=False`, raise `N_SAMPLES` to 32 or 64 for the high-k tail, add
`aime24`/`aime25` with `N_SAMPLES=64`, and move to `math-7b` (the CIR checkpoint
reproduces Fig. 6 with the control added). Per-subject breakdowns are one
`df.groupby(["tag","condition","subject"]).correct.mean()` away.

**Caveats.** `tool_on` executes model-written Python on this VM with a 10 s
timeout; fine for Colab, not for a shared machine. The CIR prompt's turn placement
is assumed. The ZeroTIR tool prompt is unverified (its repo has no card).
Qwen2.5-Math's native context is 4096, so long solutions truncate; the rate is in
the table.

In [ ]:
# Without Google Drive for desktop: pull the run to your laptop as a zip. Every JSONL row
# carries the full generation text plus call / denial / truncation counts. Uncomment, run,
# unzip locally, then  python src/grade.py <run>/<dataset>  and  python src/report.py <run>/<dataset>.
#
# from google.colab import files
# import shutil
# shutil.make_archive(f"/content/{RUN_NAME}", "zip", RESULTS, RUN_NAME)
# files.download(f"/content/{RUN_NAME}.zip")